## CNN으로 MNIST 분류하기

7-2에서 다층 퍼셉트론으로 풀었던 MNIST 분류를 8-1에서 배운 합성곱 신경망(CNN)으로 다시 풀어보기

CNN은 출처에 따라서 합성곱 층을 부르는 단위가 다를 수 있음
1. 합성곱+활성화 함수: 합성곱 층, 풀링: 풀링 층
2. 합성곱+활성화 함수+풀링: 합성곱 층&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;→ 본 수업

### 1. 모델 이해하기

```python
# 1번 레이어 : 합성곱층(Convolutional layer)
합성곱(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1) + 활성화 함수 ReLU
맥스풀링(kernel_size=2, stride=2)

# 2번 레이어 : 합성곱층(Convolutional layer)
합성곱(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1) + 활성화 함수 ReLU
맥스풀링(kernel_size=2, stride=2)

# 3번 레이어 : 전결합층(Fully-Connected layer)
특성 맵을 1차원으로 펼침  # 64 x 7 x 7 → 3136
전결합층(뉴런 10개)
```

* 출력층의 소프트맥스는 손실 함수 `nn.CrossEntropyLoss`에 포함되어 있으므로 모델에는 넣지 않음 (7-2와 같음)

In [ ]:
from IPython.display import Image

Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/12-1.convLayer1.png', width=600)

In [ ]:
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/12-1.convLayer2.png', width=600)

In [ ]:
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/12-1.convLayer3.png', width=600)

### 2. 데이터 준비

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [ ]:
# GPU 연산이 가능하다면 GPU를 사용하고, 불가능하다면 CPU를 사용
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

In [ ]:
# MNIST dataset
mnist_train = datasets.MNIST(root="Data", train=True, download=True)
mnist_test = datasets.MNIST(root="Data", train=False, download=True)

* 7-2에서는 이미지를 784 크기의 1차원 벡터로 펼쳐서 넣었지만, CNN은 공간 구조를 유지해야 하므로 **(배치 크기, 채널, 높이, 너비)** 형태로 넣음 (8-1의 9장)
    - MNIST는 흑백 이미지이므로 채널 = 1 → `view(-1, 1, 28, 28)`

In [ ]:
X = mnist_train.data.view(-1, 1, 28, 28) / 255.0 # 정규화, (배치 크기, 채널, 높이, 너비)
y = mnist_train.targets

X_test = mnist_test.data.view(-1, 1, 28, 28) / 255.0
y_test = mnist_test.targets

print(X.shape)
print(X_test.shape)

In [ ]:
train_dataset = TensorDataset(X, y)
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

test_dataset = TensorDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False, drop_last=False)

### 3. 모델 만들기

In [ ]:
model = nn.Sequential(
    # 1번 레이어: 합성곱층
    nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    # 2번 레이어: 합성곱층
    nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    # 3번 레이어: 전결합층
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 10)
).to(device)

print(model)

* `nn.Flatten()`: 특성 맵을 1차원으로 펼침 → (배치 크기, 64, 7, 7)을 (배치 크기, 3136)으로 변환
    - 7-2에서 `view(-1, 28*28)`로 이미지를 펼친 것과 같은 역할
* 전결합층의 입력 크기는 펼친 크기인 `64 * 7 * 7`과 같아야 함

#### 크기 확인

위에서 계산한 크기와 같은지 한 층씩 통과시키면서 확인

In [ ]:
x = torch.randn(1, 1, 28, 28).to(device)  # MNIST 이미지 1장과 같은 크기의 입력

for layer in model:
    x = layer(x)
    print(layer, '→', x.shape)

### 4. 학습

In [ ]:
# 오차함수 선택
loss_fn = nn.CrossEntropyLoss().to(device) # 내부적으로 softmax 포함

# 최적화 기법 선택
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# hyperparameters
num_epochs = 3

for epoch in range(num_epochs):
    for data, targets in train_dataloader:
        optimizer.zero_grad()
        y_pred = model(data.to(device))
        loss = loss_fn(y_pred, targets.to(device))
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}')

### 5. 테스트 데이터에 대해 평가

In [ ]:
model.eval()    # 신경망을 추론(학습 없이 예측만 하는 것) 모드로 전환

correct = 0

# 데이터로더에서 미니배치를 하나씩 꺼내 추론을 수행
with torch.no_grad():  # 추론 과정에는 미분이 필요없음
    for data, targets in test_dataloader:
        outputs = model(data.to(device))  # 데이터를 입력하고 출력을 계산 (128x10)

        # 추론 계산
        _, predicted = torch.max(outputs, dim=1)  # 확률이 가장 높은 레이블이 무엇인지 계산 (argmax)
        correct += (predicted == targets.to(device)).sum().item()  # 정답과 일치한 경우 정답 카운트를 증가

# 정확도 출력
data_num = len(test_dataset)  # 데이터 총 건수
print(f'테스트 데이터에서 예측 정확도: {correct}/{data_num} ({100 * correct / data_num:.0f}%)')

* 7-2의 다층 퍼셉트론(약 96%)과 정확도를 비교해 보기
    - 같은 데이터, 같은 epoch 수로 학습했지만 공간 구조 정보를 보존하는 CNN의 정확도가 더 높음

### 6. 예측 결과 확인하기

In [ ]:
import matplotlib.pyplot as plt

idx = 0  # 확인할 테스트 이미지 번호

img = X_test[idx]  # (1, 28, 28)
with torch.no_grad():
    output = model(img.view(1, 1, 28, 28).to(device))  # 이미지 1장도 (배치 크기, 채널, 높이, 너비) 형태로 넣음
_, predicted = torch.max(output, dim=1)

print('정답:', y_test[idx].item())
print('예측:', predicted.item())
plt.imshow(img.view(28, 28), cmap='gray')

### 7. Fashion-MNIST로 다시 학습해 보기

* Fashion-MNIST: 티셔츠, 바지, 신발, 가방 등 옷 이미지로 구성된 데이터셋
    - MNIST와 같이 28x28 흑백 이미지, 6만개의 훈련 데이터와 1만개의 테스트 데이터로 구성
    - 레이블: 0 부터 9 (10개), 각 번호는 옷의 종류를 의미
* `datasets.MNIST` 대신 `datasets.FashionMNIST`를 사용하면 되고, 나머지는 MNIST와 같은 방식으로 사용
* MNIST보다 어려운 데이터라서 같은 모델로 학습해도 정확도가 더 낮음

#### 1) 데이터 준비

In [ ]:
# Fashion-MNIST dataset
fashion_train = datasets.FashionMNIST(root="Data", train=True, download=True)
fashion_test = datasets.FashionMNIST(root="Data", train=False, download=True)

In [ ]:
classes = fashion_train.classes  # 레이블 번호에 해당하는 클래스 이름
print(classes)

In [ ]:
img = fashion_train.data[0]
label = fashion_train.targets[0].item()

print('레이블:', label, classes[label])
plt.imshow(img, cmap='gray')

* 앞에서와 같은 변수 이름(`X`, `y`, `train_dataloader`, `test_dataloader` 등)을 사용하므로 3~6장의 코드를 그대로 가져와 쓸 수 있음

In [ ]:
X = fashion_train.data.view(-1, 1, 28, 28) / 255.0 # 정규화, (배치 크기, 채널, 높이, 너비)
y = fashion_train.targets

X_test = fashion_test.data.view(-1, 1, 28, 28) / 255.0
y_test = fashion_test.targets

train_dataset = TensorDataset(X, y)
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

test_dataset = TensorDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False, drop_last=False)

#### 2) 직접 해보기

3~6장의 코드를 참고해서 아래를 직접 작성해 보기

1. 3장과 같은 구조의 CNN 모델을 **새로** 만들기 (위에서 MNIST로 학습한 모델을 그대로 쓰면 안 됨)
2. 손실 함수와 옵티마이저 정의
3. 학습 (`num_epochs = 3`)
4. 테스트 데이터로 평가 (정확도 약 90%)
5. 테스트 이미지의 정답과 예측을 클래스 이름으로 출력: `classes[레이블 번호]`
6. (도전) epoch 수나 `out_channels`를 바꿔서 정확도를 높여 보기

In [ ]:
# TODO: 1) CNN 모델 만들기 (3장과 같은 구조)
model = None

In [ ]:
# TODO: 2) 손실 함수와 옵티마이저 정의
loss_fn = None
optimizer = None

In [ ]:
# 3) 학습
num_epochs = 3

for epoch in range(num_epochs):
    for data, targets in train_dataloader:
        # TODO: 기울기 초기화

        # TODO: 모델의 예측값 계산 (data를 device로 보내기)
        y_pred = None

        # TODO: loss 계산 (targets를 device로 보내기)
        loss = None

        # TODO: 역전파로 기울기 계산 후 파라미터 업데이트

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}')

In [ ]:
# 4) 테스트 데이터로 평가
model.eval()    # 신경망을 추론(학습 없이 예측만 하는 것) 모드로 전환

correct = 0

with torch.no_grad():
    for data, targets in test_dataloader:
        # TODO: 모델의 출력 계산 (data를 device로 보내기)
        outputs = None

        # TODO: 확률이 가장 높은 레이블 계산 (5장처럼 torch.max 사용)

        # TODO: 정답과 일치한 개수를 correct에 더하기 (targets를 device로 보내기)

# 정확도 출력
data_num = len(test_dataset)  # 데이터 총 건수
print(f'테스트 데이터에서 예측 정확도: {correct}/{data_num} ({100 * correct / data_num:.0f}%)')

In [ ]:
# 5) 예측 결과 확인
idx = 0  # 확인할 테스트 이미지 번호

img = X_test[idx]  # (1, 28, 28)
with torch.no_grad():
    output = model(img.view(1, 1, 28, 28).to(device))

# TODO: 확률이 가장 높은 레이블 계산 (torch.max 사용)

# TODO: 정답과 예측을 클래스 이름으로 출력 (classes[레이블 번호] 사용)

plt.imshow(img.view(28, 28), cmap='gray')